# DOCX to Markdown

Manual batch conversion from `docs_input/` to `res_txt/`.

In [ ]:
from pathlib import Path
import subprocess
import tempfile


def convert(file_bytes: bytes) -> str:
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir_path = Path(tmpdir)
        docx_path = tmpdir_path / "input.docx"
        md_path = tmpdir_path / "output.md"

        docx_path.write_bytes(file_bytes)

        subprocess.run(
            ["pandoc", str(docx_path), "-t", "markdown", "--wrap=preserve", "-o", str(md_path)],
            check=True,
            text=True,
        )

        return md_path.read_text(encoding="utf-8")

In [34]:
def resolve_example_dir() -> Path:
    cwd = Path.cwd()
    if (cwd / "docs_input").exists():
        return cwd

    repo_relative_dir = Path("generator/docs/examples/docx_to_md")
    if (cwd / repo_relative_dir / "docs_input").exists():
        return cwd / repo_relative_dir

    raise FileNotFoundError("Cannot find docs_input directory")


def convert_all_docx_to_markdown() -> list[Path]:
    example_dir = resolve_example_dir()
    input_dir = example_dir / "docs_input"
    output_dir = example_dir / "res_txt"
    output_dir.mkdir(parents=True, exist_ok=True)

    docx_files = sorted(input_dir.glob("*.docx"))
    if not docx_files:
        print(f"No .docx files found in {input_dir}")
        return []

    converted_paths = []
    for input_path in docx_files:
        output_path = output_dir / f"{input_path.stem}.md"
        markdown_text = convert(input_path.read_bytes())
        output_path.write_text(markdown_text, encoding="utf-8")
        converted_paths.append(output_path)
        print(f"Converted: {input_path.name} -> {output_path}")

    print(f"Done. Converted {len(converted_paths)} file(s).")
    return converted_paths


converted_paths = convert_all_docx_to_markdown()

Converted: Средства_гигиены-res.docx -> /Users/dmitrijdeordice/Developer/Git_all_repositories/apteka/vn1_bot/generator/docs/examples/docx_to_md/res_txt/Средства_гигиены-res.md
Converted: Средства_гигиены.docx -> /Users/dmitrijdeordice/Developer/Git_all_repositories/apteka/vn1_bot/generator/docs/examples/docx_to_md/res_txt/Средства_гигиены.md
Done. Converted 2 file(s).
